# AI Programming — Lecture 22
## Variational Autoencoder (VAE) on MNIST

이번 실습에서는 **MNIST**를 이용하여 Variational Autoencoder의 핵심 구조를 직접 구현합니다.

강의에서 배운 다음 흐름을 코드로 연결하는 것이 목표입니다.

```text
Input x
   ↓
Encoder
   ↓
μ(x), log σ²(x)
   ↓
Reparameterization
   ↓
Latent z
   ↓
Decoder
   ↓
Reconstruction x_hat
```

### 학습 목표

- Autoencoder와 VAE의 차이를 설명할 수 있습니다.
- Encoder가 하나의 latent vector가 아니라 **latent distribution의 parameter**를 출력한다는 점을 이해합니다.
- Reparameterization trick을 코드로 구현할 수 있습니다.
- VAE loss를 **Reconstruction Loss + KL Divergence**로 구성할 수 있습니다.
- 2차원 latent space를 직접 시각화할 수 있습니다.
- Prior $p(z)=\mathcal{N}(0,I)$에서 sampling하여 새로운 숫자를 생성할 수 있습니다.

### 이번 실습의 구성

```text
MNIST
→ 2D Latent VAE
→ Reconstruction
→ Latent Space Visualization
→ Sampling & Generation
```

> 복잡한 고성능 모델보다 **VAE의 핵심 아이디어를 눈으로 확인하는 것**에 집중합니다.

## 0. 실습 환경 설정

MNIST와 작은 Dense network를 사용하므로 Google Colab의 CPU에서도 실행할 수 있습니다.
GPU를 사용하면 조금 더 빠릅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
LATENT_DIM = 2
BATCH_SIZE = 128
EPOCHS = 15

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

gpus = tf.config.list_physical_devices("GPU")

print("TensorFlow:", tf.__version__)
print("GPU:", gpus[0].name if gpus else "사용하지 않음 (CPU)")

## 1. MNIST Dataset

MNIST 이미지는 $28 \times 28$ grayscale image입니다.

이번 VAE에서는 decoder output을 `sigmoid`로 만들고
pixel을 $[0,1]$ 범위로 해석하므로 입력도 $[0,1]$로 scaling합니다.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Channel dimension 추가: (28, 28) → (28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("x_train:", x_train.shape)
print("x_test :", x_test.shape)
print("Pixel range:", x_train.min(), "~", x_train.max())

In [ ]:
plt.figure(figsize=(10, 2))

for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(str(y_train[i]))
    plt.axis("off")

plt.tight_layout()
plt.show()

## 2. Autoencoder와 VAE의 차이

### Autoencoder

일반 Autoencoder는 입력 $x$를 하나의 latent vector $z$로 변환합니다.

```text
x → Encoder → z → Decoder → x_hat
```

좋은 reconstruction은 가능하지만,
latent space의 빈 영역에서 아무 값이나 sampling하면
의미 없는 결과가 나올 수 있습니다.

### Variational Autoencoder

VAE의 encoder는 하나의 $z$를 직접 출력하지 않습니다.

대신 각 입력 $x$에 대해 Gaussian distribution의 parameter를 출력합니다.

$$
q_\phi(z|x)
=
\mathcal{N}
\left(
\mu(x),
\mathrm{diag}(\sigma^2(x))
\right)
$$

즉:

```text
x
↓
Encoder
├─ μ(x)
└─ log σ²(x)
```

그 distribution에서 latent vector $z$를 sampling합니다.

## 3. Reparameterization Trick

직접

$$
z \sim \mathcal{N}(\mu,\sigma^2)
$$

라고 sampling하면 stochastic sampling operation 때문에
gradient를 encoder까지 전달하기 어렵습니다.

그래서 random variable을 encoder 밖으로 분리합니다.

$$
\epsilon \sim \mathcal{N}(0,I)
$$

$$
z
=
\mu
+
\sigma \odot \epsilon
$$

코드에서는 numerical stability를 위해 $\log\sigma^2$를 사용합니다.

$$
\sigma
=
\exp
\left(
\frac{1}{2}\log\sigma^2
\right)
$$

In [ ]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs

        epsilon = tf.random.normal(
            shape=tf.shape(z_mean)
        )

        z = (
            z_mean
            + tf.exp(0.5 * z_log_var) * epsilon
        )

        return z

### 간단한 숫자 예제

같은 $\mu$와 $\sigma$에서도 $\epsilon$이 달라지면 다른 $z$가 sampling됩니다.

In [ ]:
z_mean_demo = tf.constant([[1.0, -1.0]])
z_log_var_demo = tf.constant([[0.0, 0.0]])

sampling_layer = Sampling()

for _ in range(3):
    z_demo = sampling_layer(
        [z_mean_demo, z_log_var_demo]
    )
    print(z_demo.numpy())

## 4. Encoder 만들기

이번 실습에서는 latent space를 직접 그릴 수 있도록

```text
LATENT_DIM = 2
```

를 사용합니다.

Encoder 구조:

```text
28×28×1
→ Flatten
→ Dense(256, ReLU)
→ Dense(128, ReLU)
→ μ ∈ R²
→ log σ² ∈ R²
→ Sampling
→ z ∈ R²
```

In [ ]:
encoder_inputs = keras.Input(
    shape=(28, 28, 1),
    name="encoder_input"
)

x = layers.Flatten()(encoder_inputs)
x = layers.Dense(
    256,
    activation="relu"
)(x)
x = layers.Dense(
    128,
    activation="relu"
)(x)

z_mean = layers.Dense(
    LATENT_DIM,
    name="z_mean"
)(x)

z_log_var = layers.Dense(
    LATENT_DIM,
    name="z_log_var"
)(x)

z = Sampling()(
    [z_mean, z_log_var]
)

encoder = keras.Model(
    encoder_inputs,
    [z_mean, z_log_var, z],
    name="encoder"
)

encoder.summary()

## 5. Decoder 만들기

Decoder는 latent vector $z$를 다시 image space로 변환합니다.

```text
z ∈ R²
→ Dense(128, ReLU)
→ Dense(256, ReLU)
→ Dense(784, Sigmoid)
→ 28×28×1
```

마지막 `sigmoid`는 각 pixel output을 $[0,1]$ 범위로 만듭니다.

In [ ]:
latent_inputs = keras.Input(
    shape=(LATENT_DIM,),
    name="z_sampling"
)

x = layers.Dense(
    128,
    activation="relu"
)(latent_inputs)

x = layers.Dense(
    256,
    activation="relu"
)(x)

x = layers.Dense(
    28 * 28,
    activation="sigmoid"
)(x)

decoder_outputs = layers.Reshape(
    (28, 28, 1)
)(x)

decoder = keras.Model(
    latent_inputs,
    decoder_outputs,
    name="decoder"
)

decoder.summary()

## 6. VAE Loss

VAE의 loss는 두 부분으로 구성됩니다.

$$
\mathcal{L}_{VAE}
=
\mathcal{L}_{recon}
+
\mathcal{L}_{KL}
$$

### 1) Reconstruction Loss

원본 $x$와 reconstruction $\hat{x}$가 비슷하도록 만듭니다.

MNIST pixel을 $[0,1]$ 값으로 사용하므로
이번 실습에서는 Binary Cross Entropy를 사용합니다.

$$
\mathcal{L}_{recon}
=
-\sum_i
\left[
x_i\log\hat{x}_i
+
(1-x_i)\log(1-\hat{x}_i)
\right]
$$

### 2) KL Divergence

Encoder가 만든 approximate posterior

$$
q_\phi(z|x)
$$

가 prior

$$
p(z)=\mathcal{N}(0,I)
$$

에서 너무 멀어지지 않도록 regularization합니다.

Gaussian VAE에서는 다음 식을 바로 계산할 수 있습니다.

$$
\mathcal{L}_{KL}
=
-\frac{1}{2}
\sum_j
\left(
1+\log\sigma_j^2-\mu_j^2-\sigma_j^2
\right)
$$

### 직관

```text
Reconstruction Loss
→ 입력을 잘 복원하도록 학습

KL Loss
→ latent space를 N(0, I)에 가깝게 정리
```

이 두 목적을 함께 만족시키는 것이 VAE의 핵심입니다.

## 7. VAE Model

Keras의 `train_step()`을 사용하여
reconstruction loss와 KL loss를 직접 계산합니다.

수식과 코드가 연결되는 부분을 확인하세요.

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)

        self.encoder = encoder
        self.decoder = decoder

        self.total_loss_tracker = keras.metrics.Mean(
            name="total_loss"
        )

        self.reconstruction_loss_tracker = keras.metrics.Mean(
            name="reconstruction_loss"
        )

        self.kl_loss_tracker = keras.metrics.Mean(
            name="kl_loss"
        )

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def compute_losses(self, data, training=False):
        z_mean, z_log_var, z = self.encoder(
            data,
            training=training
        )

        reconstruction = self.decoder(
            z,
            training=training
        )

        # BCE: (batch, 28, 28)
        bce = keras.losses.binary_crossentropy(
            data,
            reconstruction
        )

        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(
                bce,
                axis=(1, 2)
            )
        )

        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1
                + z_log_var
                - tf.square(z_mean)
                - tf.exp(z_log_var),
                axis=1,
            )
        )

        total_loss = (
            reconstruction_loss
            + kl_loss
        )

        return (
            total_loss,
            reconstruction_loss,
            kl_loss,
        )

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            (
                total_loss,
                reconstruction_loss,
                kl_loss,
            ) = self.compute_losses(
                data,
                training=True
            )

        gradients = tape.gradient(
            total_loss,
            self.trainable_weights
        )

        self.optimizer.apply_gradients(
            zip(
                gradients,
                self.trainable_weights
            )
        )

        self.total_loss_tracker.update_state(
            total_loss
        )

        self.reconstruction_loss_tracker.update_state(
            reconstruction_loss
        )

        self.kl_loss_tracker.update_state(
            kl_loss
        )

        return {
            "total_loss": self.total_loss_tracker.result(),
            "reconstruction_loss":
                self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        (
            total_loss,
            reconstruction_loss,
            kl_loss,
        ) = self.compute_losses(
            data,
            training=False
        )

        self.total_loss_tracker.update_state(
            total_loss
        )

        self.reconstruction_loss_tracker.update_state(
            reconstruction_loss
        )

        self.kl_loss_tracker.update_state(
            kl_loss
        )

        return {
            "total_loss": self.total_loss_tracker.result(),
            "reconstruction_loss":
                self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

## 8. VAE 학습

Training label은 사용하지 않습니다.

```text
Input  = MNIST image
Target = 같은 MNIST image
```

즉, self-reconstruction을 학습합니다.

In [ ]:
vae = VAE(
    encoder,
    decoder,
    name="vae"
)

vae.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    )
)

history = vae.fit(
    x_train,
    validation_data=x_test,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)

## 9. Loss 변화 확인

Total loss뿐 아니라
Reconstruction loss와 KL loss가 각각 어떻게 변하는지도 확인합니다.

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history.history["reconstruction_loss"],
    label="Reconstruction"
)

plt.plot(
    history.history["kl_loss"],
    label="KL"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VAE Training Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. Reconstruction

Test image를 encoder에 넣어 latent vector로 변환한 뒤
decoder로 다시 복원합니다.

```text
Original x
→ Encoder
→ z
→ Decoder
→ Reconstruction x_hat
```

VAE는 pixel-perfect reconstruction보다
**regularized latent space와 generation 가능성**도 함께 학습하므로,
일반 AE보다 reconstruction이 다소 부드럽게 보일 수 있습니다.

In [ ]:
n = 10

z_mean_test, z_log_var_test, z_test = encoder.predict(
    x_test[:n],
    verbose=0
)

reconstructed = decoder.predict(
    z_mean_test,
    verbose=0
)

plt.figure(figsize=(12, 3))

for i in range(n):
    # Original
    plt.subplot(2, n, i + 1)
    plt.imshow(
        x_test[i].squeeze(),
        cmap="gray"
    )
    plt.axis("off")

    # Reconstruction
    plt.subplot(2, n, n + i + 1)
    plt.imshow(
        reconstructed[i].squeeze(),
        cmap="gray"
    )
    plt.axis("off")

plt.suptitle(
    "Top: Original | Bottom: Reconstruction"
)

plt.tight_layout()
plt.show()

## 11. 2D Latent Space 시각화

이번 실습에서 latent dimension을 2로 둔 가장 큰 이유입니다.

Test image를 encoder에 넣고 $\mu(x)$를 2차원 평면에 표시합니다.

Label은 VAE 학습에는 사용하지 않았지만,
**시각화에서 각 숫자가 latent space의 어느 영역에 위치하는지 확인하기 위해서만 사용**합니다.

In [ ]:
z_mean_all, _, _ = encoder.predict(
    x_test,
    batch_size=256,
    verbose=0
)

plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    z_mean_all[:, 0],
    z_mean_all[:, 1],
    c=y_test,
    cmap="tab10",
    s=5,
    alpha=0.7,
)

plt.colorbar(
    scatter,
    ticks=range(10)
)

plt.xlabel("z1")
plt.ylabel("z2")
plt.title("MNIST in 2D Latent Space")
plt.grid(alpha=0.2)
plt.show()

### 확인할 내용

- 같은 숫자들이 비슷한 영역에 모이는가?
- 서로 비슷하게 생긴 숫자들의 영역이 일부 겹치는가?
- latent space가 완전히 떨어진 섬처럼만 구성되지 않고 비교적 연속적으로 연결되는가?

## 12. Prior에서 Sampling하여 새로운 숫자 생성

VAE의 generative model로서 가장 중요한 단계입니다.

Training이 끝나면 실제 image가 없어도 됩니다.

$$
z \sim p(z)=\mathcal{N}(0,I)
$$

에서 random latent vector를 sampling한 뒤 decoder에 넣습니다.

```text
Random z ~ N(0, I)
        ↓
      Decoder
        ↓
New MNIST-like image
```

In [ ]:
num_samples = 20

z_random = np.random.normal(
    loc=0.0,
    scale=1.0,
    size=(num_samples, LATENT_DIM)
).astype("float32")

generated = decoder.predict(
    z_random,
    verbose=0
)

plt.figure(figsize=(10, 4))

for i in range(num_samples):
    plt.subplot(4, 5, i + 1)
    plt.imshow(
        generated[i].squeeze(),
        cmap="gray"
    )
    plt.axis("off")

plt.suptitle(
    "Generated Samples: z ~ N(0, I)"
)

plt.tight_layout()
plt.show()

## 13. Latent Space를 Grid로 탐색하기

2차원 latent space의 여러 위치를 일정한 간격으로 선택해
decoder output을 한 번에 확인합니다.

이 그림은

> **latent 위치가 조금씩 변할 때 생성되는 숫자도 부드럽게 변하는가?**

를 확인하기 좋습니다.

In [ ]:
n = 15
grid_x = np.linspace(-2.5, 2.5, n)
grid_y = np.linspace(2.5, -2.5, n)

z_grid = np.array(
    [
        [xi, yi]
        for yi in grid_y
        for xi in grid_x
    ],
    dtype="float32"
)

decoded_grid = decoder.predict(
    z_grid,
    batch_size=256,
    verbose=0
)

digit_size = 28

figure = np.zeros(
    (
        digit_size * n,
        digit_size * n
    )
)

k = 0

for i in range(n):
    for j in range(n):
        digit = decoded_grid[k].squeeze()

        figure[
            i * digit_size:(i + 1) * digit_size,
            j * digit_size:(j + 1) * digit_size
        ] = digit

        k += 1

plt.figure(figsize=(10, 10))
plt.imshow(
    figure,
    cmap="gray"
)

plt.xlabel("z1: -2.5 → 2.5")
plt.ylabel("z2: 2.5 → -2.5")
plt.title("VAE Latent Manifold")
plt.xticks([])
plt.yticks([])
plt.show()

## 14. 선택 실습 — Latent Interpolation

두 latent point 사이를 조금씩 이동하면서
decoder output이 어떻게 변하는지 확인합니다.

이 실험은 VAE의 latent space가 **연속적이고 smooth한 구조**를 학습했는지 직관적으로 보여줍니다.

In [ ]:
# Test sample 두 개 선택
idx_a = 0
idx_b = 1

z_a = z_mean_all[idx_a]
z_b = z_mean_all[idx_b]

steps = 10

alphas = np.linspace(
    0.0,
    1.0,
    steps
)

z_interp = np.array(
    [
        (1 - a) * z_a + a * z_b
        for a in alphas
    ],
    dtype="float32"
)

interp_images = decoder.predict(
    z_interp,
    verbose=0
)

plt.figure(figsize=(12, 2))

for i in range(steps):
    plt.subplot(1, steps, i + 1)
    plt.imshow(
        interp_images[i].squeeze(),
        cmap="gray"
    )
    plt.axis("off")

plt.suptitle(
    f"Latent Interpolation: "
    f"{y_test[idx_a]} → {y_test[idx_b]}"
)

plt.tight_layout()
plt.show()

## 15. 직접 해보기

### 구조 확인

1. Encoder가 직접 $z$를 출력하지 않고 어떤 두 값을 출력하는지 확인하세요.
2. `Sampling` layer에서 reparameterization 식과 코드가 어떻게 대응하는지 찾으세요.
3. KL loss에서 `z_mean`과 `z_log_var`가 사용되는 부분을 확인하세요.

### 결과 확인

4. Original image와 reconstruction을 비교하세요.
5. 2D latent space에서 같은 digit가 비슷한 영역에 모이는지 확인하세요.
6. $z \sim \mathcal{N}(0,I)$에서 sampling한 image가 MNIST처럼 보이는지 확인하세요.
7. Latent grid에서 digit가 연속적으로 변하는지 확인하세요.

### Hyperparameter

8. `LATENT_DIM = 2`를 `10` 또는 `20`으로 바꾸면 reconstruction은 어떻게 달라질까요?
   - 단, 2D scatter와 grid visualization은 그대로 사용할 수 없습니다.
9. `EPOCHS = 5`, `15`, `30`을 비교해 보세요.
10. KL loss를 제거하면 sampling과 latent space가 어떻게 달라질지 생각해 보세요.

# 16. 정리

### Autoencoder

```text
x → Encoder → z → Decoder → x_hat
```

Latent code가 어떻게 분포해야 하는지 명시적인 제약이 없습니다.

### VAE

```text
x
↓
Encoder
↓
μ(x), log σ²(x)
↓
z = μ + σ ⊙ ε
↓
Decoder
↓
x_hat
```

### VAE Loss

$$
\mathcal{L}_{VAE}
=
\mathcal{L}_{reconstruction}
+
D_{KL}
\left(
q_\phi(z|x)
\|
p(z)
\right)
$$

### Generative Sampling

```text
z ~ N(0, I)
↓
Decoder
↓
New Sample
```

### 꼭 기억할 것

1. **VAE encoder는 latent point가 아니라 latent distribution을 학습합니다.**
2. **Reparameterization trick을 통해 sampling 과정에서도 gradient를 전달할 수 있습니다.**
3. **Reconstruction loss는 입력을 복원하도록 합니다.**
4. **KL divergence는 approximate posterior를 prior에 가깝게 regularize합니다.**
5. **학습 후에는 prior에서 z를 sampling하여 새로운 데이터를 생성할 수 있습니다.**
6. **2D latent space에서는 숫자들의 구조와 연속적인 변화를 직접 관찰할 수 있습니다.**